# S1 — 第一眼就要看的事：First Look Checklist

> **時間**：2 小時（講授 70min + 課堂練習 40min + QA 10min）  
> **資料**：`datasets/titanic/train.csv`（講授）、`datasets/house-prices/train.csv`（練習）  
> **學完能幹嘛**：拿到任何資料集，10 分鐘內建立完整的「資料概覽報告」，知道接下來該往哪個方向深入

---

## 為什麼 First Look 這麼重要？

Kaggle 比賽中，高手拿到資料後不是馬上寫模型，而是花 **至少 1 小時** 把資料「看透」。

新手最常犯的錯誤：跳過 EDA 直接跑 `RandomForest`，分數低卻不知道為什麼。

**一句話口訣：拿到資料的前 10 分鐘，決定你接下來 10 小時的方向。**

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

# 讀取 Titanic 訓練集
df = pd.read_csv('datasets/titanic/train.csv')
print(f'資料集載入完成：{df.shape[0]} 列 × {df.shape[1]} 欄')

---
## 核心觀念 1／3 — "Shape-Info-Describe" 三連發

每次拿到新資料集，**一定**先跑這三個指令。這是你的第一道防線。

| 指令 | 看什麼 | 要注意什麼 |
|:-----|:-------|:----------|
| `df.shape` | 幾列 × 幾欄 | 資料量級：百？千？百萬？ |
| `df.info()` | 每欄的型別和非空數 | 型別不對（數字存成 object）？缺值多少？ |
| `df.describe()` | 數值欄的統計摘要 | min/max 有沒有異常？mean 和 median 差很遠？ |

In [ ]:
# 第一發：shape
print(f'Shape: {df.shape}')
print(f'→ {df.shape[0]} 位乘客，{df.shape[1]} 個欄位')

In [ ]:
# 第二發：info
df.info()

**讀 `info()` 的技巧**：
- 891 筆資料，但 `Age` 只有 714 個非空值 → **177 筆缺值**（≈20%）
- `Cabin` 只有 204 個非空值 → **687 筆缺值**（≈77%，非常高）
- `Embarked` 缺 2 筆 → 影響小
- `Pclass` 是 int64，但它其實是**有序類別**（1 等 > 2 等 > 3 等），不是連續數值

In [ ]:
# 第三發：describe（加 include='all' 連類別欄位也看）
df.describe(include='all')

In [ ]:
# 加分技巧：dtype 統計
print('各型別欄位數量：')
print(df.dtypes.value_counts())

**口訣**：`shape` 看大小、`info()` 看型別和缺值、`describe(include='all')` 看分布摘要 — 三連發不能少。

> ### 💡 知識補給站 — describe() 隱藏的線索
> 
> `describe()` 裡藏著很多線索：
> - **mean 和 50% (median) 差很遠** → 資料可能嚴重偏態（右偏或左偏），例如 `Fare` 的 mean=32 但 median=14
> - **min 或 max 離 25%/75% 很遠** → 可能有離群值
> - **std 很大** → 資料分散，可能需要標準化
> - **count < 總列數** → 該欄有缺值
> - 對類別欄：**unique 很大**（接近 count）→ 可能是 ID 欄位，不適合直接做特徵
> 
> *延伸關鍵字：skewness, outlier detection, feature scaling, cardinality*

---
## 核心觀念 2／3 — Target Variable 先看

在看其他欄位之前，**一定先看目標變數 (target)**。

- 分類問題 → 用 `value_counts()` 看類別比例
- 迴歸問題 → 用 `describe()` + histogram 看分布

為什麼？因為 **target 的分布直接決定你的分析策略**：
- 類別嚴重不平衡（99:1）→ 準確率指標沒意義，要用 F1 / AUC
- 迴歸目標右偏 → 考慮 log 轉換

In [ ]:
# Titanic 的 target 是 Survived（0 或 1）
print('=== Target: Survived ===')
print(df['Survived'].value_counts())
print()
print('比例：')
print(df['Survived'].value_counts(normalize=True))

In [ ]:
# 視覺化：圓餅圖一眼看出比例
survived_counts = df['Survived'].value_counts()

labels = ['Not Survived (0)', 'Survived (1)']
colors = ['#ff9999', '#66b2ff']
explode = (0.05, 0)

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(survived_counts, explode=explode, labels=labels, colors=colors,
       autopct='%1.1f%%', startangle=90, shadow=True)
ax.set_title('Titanic Survived 比例', fontsize=14)
plt.tight_layout()
plt.show()

print('→ 存活率 38.4%，死亡率 61.6% — 不算極端不平衡，但死亡佔多數')

**口訣**：拿到資料第一件事 — 看 target 長什麼樣子。分類看 `value_counts()`，迴歸看 histogram。

---
## 核心觀念 3／3 — 欄位分類地圖 (Column Classification Map)

把所有欄位分成幾種類型，這張地圖會指引你後續每一步分析：

| 類型 | 說明 | 分析策略 |
|:-----|:-----|:--------|
| ID 識別 | 唯一值，無分析價值 | 直接排除 |
| 數值連續 | 可做加減乘除 | histogram / boxplot / 相關性 |
| 數值離散 | 看起來是數字但其實是類別 | bar chart / `crosstab` |
| 類別名義 | 沒有順序的類別 | `value_counts` / bar chart |
| 類別有序 | 有順序的類別 | 注意順序影響 |
| 文字 | 需要拆解才能用 | 字串處理（S4 會教） |

In [ ]:
# 技巧 1：用 select_dtypes 快速分組
num_cols = df.select_dtypes(include='number').columns.tolist()
obj_cols = df.select_dtypes(include='object').columns.tolist()

print(f'數值型 ({len(num_cols)} 個): {num_cols}')
print(f'物件型 ({len(obj_cols)} 個): {obj_cols}')

In [ ]:
# 技巧 2：用 nunique() 區分「真正的數值」和「偽裝成數字的類別」
print('各欄位唯一值數量：')
print(df.nunique().sort_values())

print('\n→ Survived (2), Pclass (3), SibSp (7), Parch (7) 看起來是數字')
print('  但唯一值很少，實際上應該當「類別」處理')

In [ ]:
# 建立完整的欄位分類地圖
column_map = pd.DataFrame({
    '欄位': df.columns,
    'dtype': df.dtypes.values,
    '非空數': df.notnull().sum().values,
    '缺值%': (df.isnull().mean() * 100).round(1).values,
    '唯一值': df.nunique().values,
    '範例值': [str(df[c].dropna().iloc[:3].tolist()) for c in df.columns],
})

# 手動標註欄位類型（這是分析師的判斷，不能完全自動化）
type_labels = {
    'PassengerId': 'ID',
    'Survived': 'Target (binary)',
    'Pclass': '有序類別',
    'Name': '文字',
    'Sex': '名義類別',
    'Age': '連續數值',
    'SibSp': '離散數值',
    'Parch': '離散數值',
    'Ticket': '文字 (ID-like)',
    'Fare': '連續數值',
    'Cabin': '文字 (高缺值)',
    'Embarked': '名義類別',
}
column_map['分析類型'] = column_map['欄位'].map(type_labels)
column_map

**口訣**：`select_dtypes` 先分數值和物件，`nunique()` 再區分真數值和偽數值 — 欄位分類地圖是後續所有分析的藍圖。

> ### 💡 知識補給站 — High Cardinality 陷阱
> 
> 如果一個類別欄位的唯一值接近總列數（例如 `Name` 有 891 個唯一值中的 891），這叫做 **high cardinality**。
> 
> High cardinality 欄位的問題：
> - 做 one-hot encoding 會爆炸（891 個新欄位）
> - 做 label encoding 會給模型錯誤的「順序」訊息
> - 通常不能直接當特徵，但可以**拆解**出有用資訊（例如從 Name 拆出 Title）
> 
> **判斷經驗法則**：`nunique / len(df) > 0.5` 就要小心。
> 
> *延伸關鍵字：cardinality, one-hot encoding, target encoding, feature extraction*

---
## 實務 Case — Titanic First Look 完整流程

讓我們把三個核心觀念串起來，建立一個可重用的 First Look 報告。

In [ ]:
# NA 概覽圖（S2 會深入，這裡先建立直覺）
na_pct = df.isnull().mean() * 100
na_pct = na_pct[na_pct > 0].sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
na_pct.plot(kind='bar', color='coral', ax=ax)
ax.set_title('缺值比例 (%)', fontsize=14)
ax.set_ylabel('%')
for i, v in enumerate(na_pct):
    ax.text(i, v + 1, f'{v:.1f}%', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# 缺值熱力圖 — 看缺值的「形狀」
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, ax=ax)
ax.set_title('缺值分布熱力圖（黃色 = 缺值）', fontsize=14)
plt.tight_layout()
plt.show()

print('觀察：Cabin 缺值是大面積的，Age 缺值是零散分布的 — 這暗示不同的缺失機制（S2 會深入）')

In [ ]:
# First Look 摘要
print('=' * 60)
print('Titanic First Look 摘要')
print('=' * 60)
print(f'資料量：{df.shape[0]} 列 × {df.shape[1]} 欄')
print(f'Target：Survived（存活率 {df["Survived"].mean():.1%}）')
print(f'數值欄位：{len(num_cols)} 個')
print(f'類別欄位：{len(obj_cols)} 個')
print(f'有缺值的欄位：{(df.isnull().sum() > 0).sum()} 個')
print(f'  - Cabin: {df["Cabin"].isnull().mean():.1%} 缺值（需要策略）')
print(f'  - Age:   {df["Age"].isnull().mean():.1%} 缺值（可填補）')
print(f'  - Embarked: {df["Embarked"].isnull().sum()} 筆（影響小）')
print(f'可直接排除：PassengerId（ID）, Ticket（高基數文字）')
print(f'需要特徵工程：Name（拆 Title）, Cabin（拆 Deck）')

---
## 課堂練習（40 min）

用 **House Prices** 資料集練習 First Look 三連發。

### 🟢 送分題

1. 讀取 `datasets/house-prices/train.csv`
2. 跑 `shape`、`info()`、`describe()` 三連發
3. 回答：有幾列？幾欄？有幾個數值型欄位？幾個物件型欄位？

In [ ]:
# TODO: 你的程式碼

### 🟡 核心題

1. House Prices 的 target 是 `SalePrice` — 這是分類還是迴歸問題？
2. 畫出 `SalePrice` 的 histogram（提示：`df['SalePrice'].hist(bins=30)`）
3. 用 `df.isnull().sum().sort_values(ascending=False).head(10)` 找出缺值最多的 10 個欄位
4. 在 80 個欄位中，哪些欄位的 `nunique()` 值 ≤ 5？這些是什麼類型的欄位？

In [ ]:
# TODO: 你的程式碼

### 🔴 挑戰題

寫一個 `first_look(df, target_col=None)` 函式，輸入任何 DataFrame，自動輸出：
1. 基本資訊（shape、dtype 統計）
2. 缺值摘要（只顯示有缺值的欄位）
3. 如果指定了 `target_col`，顯示 target 的分布（分類用 `value_counts`，數值用 `describe`）
4. 欄位分類建議（基於 dtype 和 nunique 的自動判斷）

用這個函式分別對 Titanic 和 House Prices 各跑一次。

In [ ]:
# TODO: 你的程式碼

---
## 小結與速查表

| 想做什麼 | 怎麼寫 |
|:---------|:-------|
| 看大小 | `df.shape` |
| 看型別與缺值 | `df.info()` |
| 看統計摘要 | `df.describe(include='all')` |
| dtype 統計 | `df.dtypes.value_counts()` |
| 類別比例 | `df['col'].value_counts(normalize=True)` |
| 唯一值數量 | `df.nunique()` |
| 分出數值欄 | `df.select_dtypes(include='number')` |
| 分出類別欄 | `df.select_dtypes(include='object')` |
| 缺值總覽 | `df.isnull().sum().sort_values(ascending=False)` |
| 缺值比例 | `df.isnull().mean() * 100` |

**下節預告 S2**：First Look 發現 Cabin 缺 77%、Age 缺 20% — 怎麼辦？刪掉？填補？→ **Missing Data Strategy**，缺值背後其實有三種不同的機制。